# SQL Single-Table Query Strategies

**Purpose:** This guide is a quick-reference strategy sheet for SQL interview questions that involve transforming, analyzing, or reshaping data within a single table. It covers the most common patterns you'll encounter — comparing rows, aggregating, ranking, handling dates, pivoting, and more — with a focus on identifying the right approach before writing any code, choosing the most efficient function, and explaining your reasoning clearly to an interviewer.

<hr style="border: 3px solid black;">

## Table of Contents

1. [First Step: Identify the Pattern](#1-first-step-identify-the-pattern)
2. [Decision Tree (5-Second Version)](#2-decision-tree-5-second-version)
3. [Pattern Library with Examples](#3-pattern-library-with-examples)
    - [A. Compare Rows](#pattern-a-compare-rows) — LAG / LEAD
    - [B. Date Operations](#pattern-b-date-operations) — EXTRACT / DATEDIFF / INTERVAL
    - [C. Combine Rows](#pattern-c-combine-rows) — GROUP BY + CASE
    - [D. Ranking](#pattern-d-ranking) — ROW_NUMBER / RANK / DENSE_RANK
    - [E. Running Totals](#pattern-e-running-totals) — SUM() OVER()
    - [F. Missing Data](#pattern-f-missing-data) — NOT EXISTS
    - [G. Deduplication](#pattern-g-deduplication) — ROW_NUMBER
    - [H. Conditional Logic](#pattern-h-conditional-logic) — CASE WHEN
    - [I. Filtering After Aggregation](#pattern-i-filtering-after-aggregation) — HAVING
    - [J. Pivoting](#pattern-j-pivoting) — CASE + GROUP BY
4. [Critical Efficiency Rules](#4-critical-efficiency-rules)
5. [GROUP BY vs Window Functions](#5-group-by-vs-window-functions)
6. [How to Think from the Vignette](#6-how-to-think-from-the-vignette)
7. [Interview Prompt Templates](#7-interview-prompt-templates)
8. [What to Avoid](#8-what-to-avoid)
9. [Final Takeaway](#9-final-takeaway)

<hr style="border: 3px solid black;">

<a id='1-first-step-identify-the-pattern'></a>

## 1. First Step: Identify the Pattern

Before writing **any** SQL, ask yourself:

> *What is this question actually asking me to do?*

The table below combines **pattern recognition**, **data type cues**, **function selection**, and **efficiency ranking** into a single reference. Functions are listed from most efficient (1) to least efficient (3).

| Pattern | Signal Words | Common Data Types | Rank | Function | Rationale |
|---|---|---|---|---|---|
| **Compare rows** | previous, next, yesterday, consecutive, following | Date/timestamp, ordered values | 1 | `LAG()` (look back) / `LEAD()` (look ahead) | Single pass; designed for ordered row comparison |
| | | | 2 | Self `JOIN` | Works but adds extra join overhead |
| | | | 3 | Correlated subquery | Re-executes for every row; slowest |
| **Date operations** | days between, month, year, age, interval, gap | Date/timestamp | 1 | `DATEDIFF()` / `DATE_PART()` / `EXTRACT()` | Built-in date math; single pass |
| | | | 2 | `INTERVAL` arithmetic (e.g., `+ INTERVAL '1 day'`) | Flexible but syntax varies by dialect |
| | | | 3 | Manual cast/subtraction | Error-prone; avoid when date functions exist |
| **Combine rows** | avg, sum, duration, per X | Numeric, categorical (start/end), IDs | 1 | `GROUP BY` (+ `CASE`) | Direct aggregation; no duplicate rows produced |
| | | | 2 | Self `JOIN` | Clean pairing but more work than GROUP BY |
| | | | 3 | Window function | Creates duplicates that need removal |
| **Rank rows** | top, latest, first, nth | Ordered values, date/timestamp | 1 | `ROW_NUMBER()` | Direct ranking with partition control |
| | | | 2 | `RANK()` / `DENSE_RANK()` | Use when ties matter; slightly more complex |
| | | | 3 | Correlated subquery | Runs per row; poor performance on large tables |
| **Running calc** | cumulative, rolling, moving | Numeric, date/timestamp | 1 | `SUM() OVER()` | Efficient window frame; single pass |
| | | | 2 | Self `JOIN` | Joins all preceding rows; heavier |
| | | | 3 | Correlated subquery | Re-sums for every row; very slow |
| **Existence** | no, missing, without, never | IDs, any type | 1 | `NOT EXISTS` | Handles NULLs correctly; stops at first match |
| | | | 2 | `LEFT JOIN` + `IS NULL` | Valid anti-join; slightly more verbose |
| | | | 3 | `NOT IN` | Fails silently when NULLs present in subquery |
| **Deduplicate** | unique, one per, most recent | IDs, date/timestamp | 1 | `ROW_NUMBER()` | Full control over which row to keep |
| | | | 2 | `DISTINCT` | Simple but no control over row selection |
| | | | 3 | Self `JOIN` | Overly complex for deduplication |
| **Conditional logic** | label, categorize, bucket, if/then, flag | Any type | 1 | `CASE WHEN` (standalone) | Row-level labeling; no aggregation needed |
| | | | 2 | `CASE` inside `COUNT` / `SUM` | Count or sum by category in one query |
| | | | 3 | Multiple queries with `WHERE` | Separate query per category; redundant work |
| **Filter groups** | more than, at least, groups where, having | Numeric (aggregated) | 1 | `GROUP BY` + `HAVING` | Filters groups after aggregation; designed for this |
| | | | 2 | Subquery + `WHERE` | Works but adds nesting; less readable |
| **Pivot** | rows to columns, side by side, crosstab | Categorical + numeric | 1 | `CASE` + `GROUP BY` (manual pivot) | Portable; works in all dialects |
| | | | 2 | `PIVOT` keyword (SQL Server / Oracle) | Cleaner syntax but dialect-specific |
| | | | 3 | Application-level pivot | Moves work outside SQL; last resort |

<hr style="border: 3px solid black;">

<a id='2-decision-tree-5-second-version'></a>

## 2. Decision Tree (5-Second Version)

Use this quick mental checklist when you first read a problem:

```
┌───────────────────────────────────────┐
│        READ THE QUESTION              │
│        What am I being asked to do?   │
└──────────────────┬────────────────────┘
                   │
                   ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Does it involve dates?      ├───────►│  See DATE FORK below         │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I comparing rows?        ├───────►│  LAG() / LEAD()              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I collapsing rows?       ├───────►│  GROUP BY                    │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Filtering after grouping?   ├───────►│  GROUP BY + HAVING           │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I ranking rows?          ├───────►│  ROW_NUMBER() / RANK         │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Running calculation?        ├───────►│  SUM() OVER()                │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Labeling or categorizing?   ├───────►│  CASE WHEN                   │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Turning rows into columns?  ├───────►│  CASE + GROUP BY (pivot)     │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Finding missing data?       ├───────►│  NOT EXISTS                  │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Removing duplicates?        ├───────►│  ROW_NUMBER / DISTINCT       │
└──────────────────────────────┘        └──────────────────────────────┘
```

### DATE FORK — What kind of date work?

```
┌──────────────────────────────────────────────────────────────────────┐
│                    What kind of date work?                           │
└──────┬───────────────────────┬───────────────────────┬───────────────┘
       ▼                       ▼                       ▼
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  Comparing   │        │  Extracting  │        │  Calculating │
│  consecutive │        │  parts?      │        │  difference? │
│  rows?       │        │              │        │              │
│  (yesterday, │        │  (per month, │        │  (days       │
│   next day)  │        │   per year)  │        │   between,   │
│              │        │              │        │   gaps)      │
└──────┬───────┘        └──────┬───────┘        └──────┬───────┘
       ▼                       ▼                       ▼
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  LAG / LEAD  │        │  EXTRACT /   │        │  DATEDIFF /  │
│  + date math │        │  DATE_PART   │        │  subtraction │
│              │        │  + GROUP BY  │        │  + LAG/LEAD  │
└──────────────┘        └──────────────┘        └──────────────┘
```

**Key insight:** Date problems almost always combine with another pattern. The date fork tells you *which* date tool you need, then you still flow into the main tree for the structural pattern (GROUP BY, LAG, ROW_NUMBER, etc.).

<hr style="border: 3px solid black;">

<a id='3-pattern-library-with-examples'></a>

## 3. Pattern Library with Examples

<a id='pattern-a-compare-rows'></a>

### A. Compare Rows

**Signals:** previous, yesterday, next, consecutive, following

---

**Best approach — Window Function: `LAG()` (look backward)**

**Method:** `LAG()` is a window function that lets you access the previous row's value without a join. You define the order with `OVER (ORDER BY ...)` and it hands you the value from the row right before the current one. Use `LAG()` when the question asks about what came before — "previous day," "yesterday," "prior month."

**In plain language:** The inner query adds two new columns to every row — the previous day's date and the previous day's temperature. Then the outer query simply checks: "Is today exactly one day after yesterday, and is today's temperature higher?" If both are true, we keep that row's ID.

```sql
SELECT id
FROM (
    SELECT
        id,
        recordDate,
        temperature,
        LAG(recordDate) OVER (ORDER BY recordDate) AS prev_date,
        LAG(temperature) OVER (ORDER BY recordDate) AS prev_temp
    FROM Weather
) t
WHERE recordDate = prev_date + INTERVAL '1 day'
  AND temperature > prev_temp;
```

---

**Also common — Window Function: `LEAD()` (look forward)**

**Method:** `LEAD()` is the mirror image of `LAG()` — instead of looking at the previous row, it looks at the next row. Use `LEAD()` when the question asks about what comes after — "next purchase," "following day," "will the customer return."

**In plain language:** For every row, we peek ahead at the next row's value. This is useful when you need to ask "what happens next?" For example, finding users whose next login was more than 30 days later — that gap tells you they churned.

```sql
SELECT
    user_id,
    login_date,
    LEAD(login_date) OVER (PARTITION BY user_id ORDER BY login_date) AS next_login,
    LEAD(login_date) OVER (PARTITION BY user_id ORDER BY login_date) - login_date AS days_until_next
FROM Logins;
```

---

**When to use LAG vs LEAD**

| Question asks about... | Use | Example |
|---|---|---|
| What came **before** this row | `LAG()` | "Was yesterday's temperature lower?" |
| What comes **after** this row | `LEAD()` | "When is the user's next login?" |
| The **gap** between consecutive rows | Either works | "How many days between orders?" |

**In plain language:** Think of it like standing in a line. `LAG()` lets you turn around and look at the person behind you. `LEAD()` lets you look at the person in front of you. Both are window functions, both use `OVER (ORDER BY ...)`, and the syntax is identical — the only difference is the direction.

---

**Alternative — Self Join**

**Method:** Join the table to itself by matching each row to the row from the day before. No window function needed, but the database has to do extra work to pair up the rows.

**In plain language:** We take two copies of the same Weather table and line them up so that each day in the first copy is matched to the day before it in the second copy. Then we just check which days were warmer than the day before.

```sql
SELECT w1.id
FROM Weather w1
JOIN Weather w2
  ON w1.recordDate = w2.recordDate + INTERVAL '1 day'
WHERE w1.temperature > w2.temperature;
```

<hr style="border: 2px solid black;">

<a id='pattern-b-date-operations'></a>

### B. Date Operations

**Signals:** days between, month, year, age, interval, gap, extract

Date columns show up constantly in single-table interview problems. These functions are often **combined** with other patterns (LAG, GROUP BY, ROW_NUMBER) rather than used alone.

---

**Extracting parts of a date — `EXTRACT()` / `DATE_PART()` / `YEAR()`, `MONTH()`, `DAY()`**

**Method:** These functions pull a specific component (year, month, day, hour) out of a date or timestamp column. The syntax varies by dialect but the idea is the same everywhere.

**In plain language:** If you have a column with full dates like `2024-03-15` and the question asks "how many orders per month," you need to break that date into its month component first, then group by it. Think of it like opening a date and pulling out just the piece you need.

```sql
-- PostgreSQL / standard SQL
SELECT EXTRACT(MONTH FROM order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY EXTRACT(MONTH FROM order_date);

-- MySQL
SELECT MONTH(order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY MONTH(order_date);

-- SQL Server
SELECT DATEPART(MONTH, order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY DATEPART(MONTH, order_date);
```

---

**Calculating differences between dates — `DATEDIFF()` / subtraction / `AGE()`**

**Method:** These calculate the gap between two dates. Often paired with `LAG()` or `LEAD()` to find the time between consecutive rows in the same table. Syntax varies heavily by dialect.

**In plain language:** When you need to answer "how many days between event A and event B," you need date difference. If both events live in the same row (like a start and end column), you just subtract. If they live in different rows (like consecutive logins), you first use `LAG()` or `LEAD()` to bring them onto the same row, then subtract.

```sql
-- PostgreSQL: subtract dates directly (returns integer days)
SELECT order_date - LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS days_since_last
FROM Orders;

-- MySQL: use DATEDIFF
SELECT DATEDIFF(order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)) AS days_since_last
FROM Orders;

-- SQL Server: use DATEDIFF with unit
SELECT DATEDIFF(DAY, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date), order_date) AS days_since_last
FROM Orders;
```

---

**Date arithmetic — `INTERVAL` / `DATE_ADD()` / `DATEADD()`**

**Method:** Add or subtract a specific amount of time from a date. Used when the question says things like "within 7 days" or "one day after."

**In plain language:** This is for when you need to shift a date forward or backward by a fixed amount — like checking if a row's date is exactly one day after another, or finding all records from the last 30 days.

```sql
-- PostgreSQL
WHERE recordDate = prev_date + INTERVAL '1 day'

-- MySQL
WHERE recordDate = DATE_ADD(prev_date, INTERVAL 1 DAY)

-- SQL Server
WHERE recordDate = DATEADD(DAY, 1, prev_date)
```

---

**Common interview pattern: dates + LAG/LEAD together**

Most date questions in interviews aren't purely about date math — they combine dates with row comparison. The date functions handle the "how far apart" part, and LAG/LEAD handle the "bring two rows together" part.

| Interview question sounds like... | Approach |
|---|---|
| "Days between consecutive logins" | `LAG()` + date subtraction |
| "Orders per month" | `EXTRACT(MONTH ...)` + `GROUP BY` |
| "Users inactive for 30+ days" | `LEAD()` + `DATEDIFF()` |
| "Was the previous day exactly yesterday?" | `LAG()` + `INTERVAL '1 day'` |
| "First order each year" | `EXTRACT(YEAR ...)` + `ROW_NUMBER()` |

<hr style="border: 2px solid black;">

<a id='pattern-c-combine-rows'></a>

### C. Combine Rows

**Signals:** average, duration, per X

---

**Best approach — Aggregate Function: `GROUP BY` + Conditional Aggregation with `CASE`**

**Method:** `GROUP BY` collapses multiple rows into one row per group. Inside the aggregation, `CASE` expressions act like an if/then switch — they pick out specific values (like the "start" timestamp vs the "end" timestamp) so you can combine them in one pass.

**In plain language:** The inner query groups every activity by machine and process, then uses `CASE` to grab the end timestamp and the start timestamp separately, and subtracts them to get the process time. The outer query then takes all those process times for each machine and averages them. Two levels of grouping, but the logic reads top to bottom: first get each process duration, then average them per machine.

```sql
SELECT
    machine_id,
    ROUND(AVG(process_time)::numeric, 3)
FROM (
    SELECT
        machine_id,
        process_id,
        MAX(CASE WHEN activity_type = 'end' THEN timestamp END) -
        MAX(CASE WHEN activity_type = 'start' THEN timestamp END) AS process_time
    FROM Activity
    GROUP BY machine_id, process_id
) t
GROUP BY machine_id;
```

<hr style="border: 2px solid black;">

<a id='pattern-d-ranking'></a>

### D. Ranking

**Signals:** top, latest, first, nth

---

**Best approach — Window Function: `ROW_NUMBER()`**

**Method:** `ROW_NUMBER()` is a window function that assigns a sequential number to each row within a partition. `PARTITION BY` splits the data into groups (like one group per customer), and `ORDER BY` controls which row gets number 1 within each group.

**In plain language:** The inner subquery is adding a row number to every row for a given customer_id, starting with the latest date as number 1, the second latest as number 2, and so on. So when we get to the outer query and filter for `rn = 1`, we are keeping only the most recent order for each customer and throwing away all the older ones.

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY order_date DESC
           ) AS rn
    FROM Orders
) t
WHERE rn = 1;
```

---

**When to use `RANK()` or `DENSE_RANK()` instead**

**Method:** `RANK()` and `DENSE_RANK()` are also window functions, but they handle ties differently. `RANK()` skips numbers after a tie (1, 1, 3), while `DENSE_RANK()` does not skip (1, 1, 2). Use these when the problem says "top N" and ties should be included.

**In plain language:** If two customers placed orders on the exact same date and you want to keep both of them (not randomly pick one), use `RANK()` instead of `ROW_NUMBER()`. The rest of the query stays the same — just swap the function name.

<hr style="border: 2px solid black;">

<a id='pattern-e-running-totals'></a>

### E. Running Totals

**Signals:** cumulative, rolling, moving

---

**Best approach — Window Function: `SUM() OVER()`**

**Method:** `SUM() OVER(ORDER BY ...)` is a window function that computes a running (cumulative) sum. Because there is no `PARTITION BY`, it treats the entire result set as one group. The `ORDER BY` inside `OVER()` tells SQL to add up all the sales from the beginning up to the current row.

**In plain language:** For every row, SQL looks at all the rows from the top of the table down to the current row (based on date order) and adds up their sales values. So row 1 shows just day 1's sales, row 2 shows day 1 + day 2, row 3 shows day 1 + day 2 + day 3, and so on. You get a running total that grows as you go down the table, and every original row is still there.

```sql
SELECT
    date,
    SUM(sales) OVER (ORDER BY date) AS running_sales
FROM DailySales;
```

<hr style="border: 2px solid black;">

<a id='pattern-f-missing-data'></a>

### F. Missing Data

**Signals:** no, missing, without, never

---

**Best approach — Subquery Filter: `NOT EXISTS`**

**Method:** `NOT EXISTS` is a subquery filter (not a window function). It checks whether a matching row exists in another table. If no match is found, the row from the outer query is kept. It stops searching as soon as it finds the first match, making it efficient.

**In plain language:** We start with every customer in the Customers table. For each one, we peek into the Orders table and ask "does this customer have at least one order?" If the answer is no — meaning `NOT EXISTS` found zero matching rows — then we keep that customer in our results. This gives us all the customers who have never placed an order.

```sql
SELECT c.customer_id
FROM Customers c
WHERE NOT EXISTS (
    SELECT 1
    FROM Orders o
    WHERE o.customer_id = c.customer_id
);
```

---

**Why NOT EXISTS over NOT IN**

`NOT IN` fails silently when NULLs are present in the subquery — if even one NULL exists in the list, the entire `NOT IN` check returns no results. `NOT EXISTS` handles NULLs correctly because it only checks whether a matching row exists, regardless of NULL values.

<hr style="border: 2px solid black;">

<a id='pattern-g-deduplication'></a>

### G. Deduplication

**Signals:** unique, one per, most recent per

---

**Best approach — Window Function: `ROW_NUMBER()`**

**Method:** Same window function as ranking — `ROW_NUMBER()` with `PARTITION BY` and `ORDER BY`. The difference is intent: here we are not ranking for display, we are numbering rows so we can throw away the duplicates and keep only the one we want (usually the most recent).

**In plain language:** The inner subquery looks at all login records for each user and numbers them starting from the most recent login as 1, the second most recent as 2, and so on. The outer query then says "only give me the rows numbered 1" — which means for every user, we keep just their latest login and discard all the older ones. This is the go-to pattern anytime you need "one row per [something]."

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY user_id
               ORDER BY login_time DESC
           ) AS rn
    FROM Logins
) t
WHERE rn = 1;
```

<hr style="border: 2px solid black;">

<a id='pattern-h-conditional-logic'></a>

### H. Conditional Logic (CASE Expressions)

**Signals:** label, categorize, bucket, if/then, classify, flag, convert rows to columns

---

**Standalone CASE — Categorizing or Bucketing Values**

**Method:** `CASE WHEN ... THEN ... ELSE ... END` is SQL's if/then/else. It evaluates conditions row by row and returns a value based on the first match. Use it standalone in SELECT to create new label columns, or inside WHERE/ORDER BY for conditional filtering and sorting.

**In plain language:** You're looking at each row and sticking a label on it. If the score is above 90, call it "high." If it's between 50 and 90, call it "medium." Everything else is "low." The table stays the same size — you're just adding a new column with your labels.

```sql
SELECT
    user_id,
    score,
    CASE
        WHEN score >= 90 THEN 'high'
        WHEN score >= 50 THEN 'medium'
        ELSE 'low'
    END AS score_tier
FROM Users;
```

---

**CASE inside COUNT / SUM — Counting by Category**

**Method:** Wrapping `CASE` inside an aggregate function like `COUNT()` or `SUM()` lets you count or sum only the rows that match a condition. This is a very common interview pattern for getting counts of different categories in a single query.

**In plain language:** Instead of running three separate queries to count how many users are "high," "medium," and "low," you do it all at once. For each row, `CASE` asks "does this match?" — if yes, it returns 1 (which gets counted), if no, it returns NULL (which gets skipped).

```sql
SELECT
    department,
    COUNT(CASE WHEN status = 'active' THEN 1 END) AS active_count,
    COUNT(CASE WHEN status = 'inactive' THEN 1 END) AS inactive_count,
    COUNT(*) AS total_count
FROM Employees
GROUP BY department;
```

<hr style="border: 2px solid black;">

<a id='pattern-i-filtering-after-aggregation'></a>

### I. Filtering After Aggregation (HAVING)

**Signals:** groups with more than, only customers who bought at least, categories where the average exceeds

---

**HAVING — The WHERE Clause for Groups**

**Method:** `HAVING` filters rows *after* `GROUP BY` has collapsed them. `WHERE` filters individual rows before grouping; `HAVING` filters the groups themselves based on aggregate values. You cannot use `WHERE` to filter on `COUNT()`, `SUM()`, `AVG()`, etc. — that's what `HAVING` is for.

**In plain language:** First, GROUP BY creates one row per group. Then HAVING looks at each group and asks "does this group meet my condition?" For example, "only keep customers who have more than 3 orders." WHERE can't do this because at the time WHERE runs, the rows haven't been grouped yet — it doesn't know what the count is.

```sql
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM Orders
GROUP BY customer_id
HAVING COUNT(*) > 3;
```

---

**WHERE vs HAVING — When to Use Which**

| Clause | Filters on | Runs when | Example |
|---|---|---|---|
| `WHERE` | Individual rows (before grouping) | Before `GROUP BY` | `WHERE status = 'active'` |
| `HAVING` | Aggregated groups (after grouping) | After `GROUP BY` | `HAVING COUNT(*) > 3` |

**In plain language:** Think of it as two gates. The first gate (WHERE) decides which individual rows are allowed into the grouping party. The second gate (HAVING) looks at the groups that formed and decides which groups are allowed into the final result. You can use both in the same query — WHERE narrows the rows first, then GROUP BY collapses them, then HAVING narrows the groups.

```sql
-- "Active customers who placed more than 5 orders in 2024"
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM Orders
WHERE status = 'completed'                        -- gate 1: only completed orders
  AND EXTRACT(YEAR FROM order_date) = 2024        -- gate 1: only 2024
GROUP BY customer_id
HAVING COUNT(*) > 5;                              -- gate 2: only groups with 5+
```

<hr style="border: 2px solid black;">

<a id='pattern-j-pivoting'></a>

### J. Pivoting (Rows to Columns)

**Signals:** show each category as a column, side by side, pivot, crosstab, transpose

---

**Pivoting with CASE + GROUP BY — Turning Rows into Columns**

**Method:** Combine `CASE` expressions inside aggregate functions with `GROUP BY` to transform row values into separate columns. This is the most portable approach — it works in every SQL dialect. Some dialects also have a dedicated `PIVOT` keyword, but the CASE approach is what interviewers typically expect.

**In plain language:** Imagine you have a table where each row is a student's score for a different subject (one row for Math, one for Science, one for English). The interviewer wants you to show one row per student with Math, Science, and English as separate columns. You use CASE to say "if the subject is Math, grab the score" and wrap it in MAX or SUM so each CASE produces one value per student.

```sql
SELECT
    student_id,
    MAX(CASE WHEN subject = 'Math' THEN score END) AS math_score,
    MAX(CASE WHEN subject = 'Science' THEN score END) AS science_score,
    MAX(CASE WHEN subject = 'English' THEN score END) AS english_score
FROM Scores
GROUP BY student_id;
```

---

**Why MAX() around the CASE?**

**In plain language:** After GROUP BY collapses the rows, each group might have multiple rows (one per subject). The CASE picks the score only when the subject matches and returns NULL for the rest. MAX() grabs that one non-NULL value from the group. You could also use MIN() or SUM() — it doesn't matter when there's only one non-NULL value per group. The point is you need *some* aggregate function to satisfy GROUP BY.

---

**COUNT(DISTINCT ...) — Counting Unique Values in Groups**

**Signals:** how many different, number of unique, distinct count per group

**Method:** `COUNT(DISTINCT column)` inside a GROUP BY counts only the unique values of that column within each group. Regular `COUNT(*)` counts all rows; `COUNT(DISTINCT ...)` deduplicates before counting.

**In plain language:** If a customer bought the same product 3 times, `COUNT(*)` would say 3 orders but `COUNT(DISTINCT product_id)` would say 1 unique product. This comes up when the question asks "how many *different* products" rather than "how many orders."

```sql
SELECT
    customer_id,
    COUNT(*) AS total_orders,
    COUNT(DISTINCT product_id) AS unique_products
FROM Orders
GROUP BY customer_id;
```

<hr style="border: 3px solid black;">

<a id='4-critical-efficiency-rules'></a>

## 4. Critical Efficiency Rules

### Rule 1 — Avoid Duplicates

| Bad | Good |
|---|---|
| Window + `DISTINCT` | `GROUP BY` |

Using `DISTINCT` after a window function means the window computed values for rows that will be thrown away. Use `GROUP BY` to aggregate upfront.

### Rule 2 — Avoid Repeated Work

| Bad | Good |
|---|---|
| Correlated subquery | Window / Join |

Correlated subqueries re-execute for every row in the outer query. Window functions and joins compute results in a single pass.

### Rule 3 — Match Tool to Structure

| Need | Use |
|---|---|
| Previous row | `LAG` |
| One row per group | `GROUP BY` |
| Keep all rows | Window function |
| Pair rows | Join |

<hr style="border: 3px solid black;">

<a id='5-group-by-vs-window-functions'></a>

## 5. GROUP BY vs Window Functions

This is one of the **most important concepts** to understand for SQL interviews.

| Feature | GROUP BY | Window Function |
|---|---|---|
| **Effect on rows** | Collapses (reduces) rows | Keeps all rows |
| **Output** | 1 row per group | All original rows + new column |
| **Use when** | You need aggregated results only | You need row-level + group-level data |

### Example: GROUP BY Result

| machine | avg_time |
|---|---|
| A | 3.5 |
| B | 2.1 |

### Example: Window Function Result

| machine | process | time | avg_time |
|---|---|---|---|
| A | 1 | 3.0 | 3.5 |
| A | 2 | 4.0 | 3.5 |
| B | 1 | 2.1 | 2.1 |

### Memory Rule

| Rule | Meaning |
|---|---|
| Look across rows → **Window** | Comparisons |
| Collapse rows → **GROUP BY** | Aggregation |
| Pair rows → **Join** | Start/end matching |
| Keep rows + add info → **Window** | Group context |
| Need anti-match → **NOT EXISTS** | Missing data |

<hr style="border: 3px solid black;">

<a id='6-how-to-think-from-the-vignette'></a>

## 6. How to Think from the Vignette

Follow these steps **before writing any SQL**:

### Step 1 — Identify the Output Shape

> *What should the final table look like?*

Examples:
- Weather problem → list of IDs (one per qualifying day)
- Activity problem → 1 row per machine

### Step 2 — Ask: Row vs Group?

> *Am I comparing rows or combining rows?*

- Comparing → Window function territory
- Combining → GROUP BY territory

### Step 3 — Identify the Grouping Level

> *What is the unit of output?*

Examples:
- Per machine → `GROUP BY machine_id`
- Per customer → `GROUP BY customer_id`
- No grouping → no `GROUP BY`

### Step 4 — Identify Pairing Logic

> *Do I need to match rows together?*

Examples:
- Start + end → `CASE` aggregation or self join
- Today + yesterday → `LAG`

### Step 5 — Choose the Tool

Use the [Decision Tree](#2-decision-tree-5-second-version) and the [Master Pattern Table](#1-first-step-identify-the-pattern) to select the best approach.

---

### Worked Examples

**Weather Problem:**

| Step | Answer |
|---|---|
| Output shape | List of IDs |
| Row vs group | Comparing rows |
| Grouping level | None |
| Pairing logic | Today vs yesterday |
| **Tool** | **LAG** |

**Activity Problem:**

| Step | Answer |
|---|---|
| Output shape | 1 row per machine |
| Row vs group | Combining rows |
| Grouping level | machine_id, process_id |
| Pairing logic | Start + end timestamps |
| **Tool** | **GROUP BY + CASE** |

<hr style="border: 3px solid black;">

<a id='7-interview-prompt-templates'></a>

## 7. Interview Prompt Templates

Use these phrases during an interview to structure your thinking out loud:

| # | Template | When to Use |
|---|---|---|
| 1 | *"This is a [compare/combine/rank] problem because..."* | Pattern identification |
| 2 | *"The output is one row per [X], so I'll group by [X]."* | Output shape |
| 3 | *"I need to compare each row to the previous one, so I'll use LAG."* | Row comparison |
| 4 | *"I need to collapse rows into one, so I'll use GROUP BY."* | Aggregation |
| 5 | *"I need to match related rows, so I'll join or use CASE aggregation."* | Pairing rows |
| 6 | *"This avoids duplicate computation and reduces rows early."* | Efficiency reasoning |
| 7 | *"Window functions let me keep all rows while adding group-level values."* | Window explanation |
| 8 | *"I won't use DISTINCT because it removes duplicates after computation."* | Avoiding mistakes |
| 9 | *"Does this produce exactly one row per required output?"* | Final check |
| 10 | *"If window functions weren't available, I'd use a self join."* | Backup strategy |

<hr style="border: 3px solid black;">

<a id='8-what-to-avoid'></a>

## 8. What to Avoid

| Bad Pattern | Why It's Bad | Better Alternative |
|---|---|---|
| Window + `DISTINCT` | Duplicate work — computes then discards | `GROUP BY` |
| Correlated subquery | Repeated work — runs per row | `LAG` / `JOIN` |
| `NOT IN` (with NULLs) | Incorrect results — NULLs break logic | `NOT EXISTS` |
| Self join for running totals | Heavy computation | `SUM() OVER()` |

<hr style="border: 3px solid black;">

<a id='9-final-takeaway'></a>

## 9. Final Takeaway

The entire game is:

```
Identify the shape  →  Pick the tool  →  Avoid unnecessary work
```

### Interview Script (Memorize This)

> *"I identify whether this is comparing rows or aggregating them. If it's row comparison, I use LAG. If it's aggregation, I use GROUP BY. If I need both row-level and group-level data, I use a window function."*